In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
from xgboost import XGBRegressor
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings("ignore")
from catboost import CatBoostRegressor

In [2]:
df = pd.read_csv("train.csv")
df_test = pd.read_csv("test.csv")
target = df.columns.tolist()[-1]
print(df.shape)
df.head()

(517754, 14)


,id,road_type,num_lanes,curvature,speed_limit,lighting,weather,road_signs_present,public_road,time_of_day,holiday,school_season,num_reported_accidents,accident_risk
0,0,urban,2,0.06,35,daylight,rainy,False,True,afternoon,False,True,1,0.13
1,1,urban,4,0.99,35,daylight,clear,True,False,evening,True,True,0,0.35
2,2,rural,4,0.63,70,dim,clear,False,True,morning,True,False,2,0.30
3,3,highway,4,0.07,35,dim,rainy,True,True,morning,False,False,1,0.21
4,4,rural,1,0.58,60,daylight,foggy,False,False,evening,True,False,1,0.56


In [3]:
def create_frequency_features(train_df, test_df, cols, num, cat):
    """
    Add frequency and binning features to the dataset.
    
    - For each column, create <col>_freq = how often each value appears in train data.
    - For numeric columns, split values into 5 and 10 quantile bins (groups) to show rank or range.
    """
    train, test = train_df.copy(), test_df.copy()

    for col in cols:
        # Frequency encoding: how common each value is
        freq = train[col].value_counts(normalize=True)
        train[f"{col}_freq"] = train[col].map(freq)
        test[f"{col}_freq"] = test[col].map(freq).fillna(train[f"{col}_freq"].mean())

        # Binning: group numeric values into quantiles
        if col in num:
            for q in [5, 10, 15]:
                try:
                    train[f"{col}_bin{q}"], bins = pd.qcut(train[col], q=q, labels=False, retbins=True, duplicates="drop")
                    test[f"{col}_bin{q}"] = pd.cut(test[col], bins=bins, labels=False, include_lowest=True)
                except Exception:
                    train[f"{col}_bin{q}"] = test[f"{col}_bin{q}"] = 0

    new_num = train.drop(columns=cat+[target]).columns.tolist()
    return train, test, new_num

In [4]:
# Identify feature
cols = df.drop(columns=target).columns.tolist()

# Categorical features
cat = [col for col in cols if df[col].dtype in ["object","category"] and col != target]

# Numerical features
num = [col for col in cols if df[col].dtype not in ["object","category","bool"] and col not in ["id", target]]

# Creating new features based on the frequency of numerical features
df, df_test, new_num = create_frequency_features(df, df_test.copy(), cols, num, cat)

# Preparing categorical features
df[cat], df_test[cat] = df[cat].astype("category"), df_test[cat].astype("category")

# Mapping a column
map_col = "num_reported_accidents"
map_num_reported = {0:0, 1:0, 2:0, 3:2, 4:4, 5:3, 6:1, 7:0}
df[map_col] = df[map_col].map(map_num_reported)
df_test[map_col] = df_test[map_col].map(map_num_reported)

# Dropping unnecessary columns
remove = ["time_of_day", "num_lanes", "road_type", "road_signs_present", "id_freq"]
df = df.drop(columns=remove)
df_test = df_test.drop(columns=remove)

# Dropping ID and duplicates
df.drop(columns="id", inplace=True)
df.drop_duplicates(inplace=True)

In [5]:
print(df.columns.tolist())

['curvature', 'speed_limit', 'lighting', 'weather', 'public_road', 'holiday', 'school_season', 'num_reported_accidents', 'accident_risk', 'road_type_freq', 'num_lanes_freq', 'num_lanes_bin5', 'num_lanes_bin10', 'num_lanes_bin15', 'curvature_freq', 'curvature_bin5', 'curvature_bin10', 'curvature_bin15', 'speed_limit_freq', 'speed_limit_bin5', 'speed_limit_bin10', 'speed_limit_bin15', 'lighting_freq', 'weather_freq', 'road_signs_present_freq', 'public_road_freq', 'time_of_day_freq', 'holiday_freq', 'school_season_freq', 'num_reported_accidents_freq', 'num_reported_accidents_bin5', 'num_reported_accidents_bin10', 'num_reported_accidents_bin15']


In [6]:
df.head()

,curvature,speed_limit,lighting,weather,public_road,holiday,school_season,num_reported_accidents,accident_risk,road_type_freq,...,weather_freq,road_signs_present_freq,public_road_freq,time_of_day_freq,holiday_freq,school_season_freq,num_reported_accidents_freq,num_reported_accidents_bin5,num_reported_accidents_bin10,num_reported_accidents_bin15
0,0.06,35,daylight,rainy,True,False,True,0,0.13,0.330974,...,0.303204,0.500796,0.502256,0.331252,0.496502,0.497514,0.404968,0,0,0
1,0.99,35,daylight,clear,False,True,True,0,0.35,0.330974,...,0.346315,0.499204,0.497744,0.333821,0.503498,0.497514,0.241947,0,0,0
2,0.63,70,dim,clear,True,True,False,0,0.30,0.333593,...,0.346315,0.500796,0.502256,0.334927,0.503498,0.502486,0.281920,1,1,1
3,0.07,35,dim,rainy,True,False,False,0,0.21,0.335433,...,0.303204,0.499204,0.502256,0.334927,0.496502,0.502486,0.404968,0,0,0
4,0.58,60,daylight,foggy,False,True,False,0,0.56,0.333593,...,0.350481,0.500796,0.497744,0.333821,0.503498,0.502486,0.404968,0,0,0


In [7]:
# Prepare DMatrix for XGBoost
dtrain = xgb.DMatrix(df.drop(columns=target), label=df[target], enable_categorical=True)

# Define XGBoost parameters
xgb_params = {
    'max_depth': 11, 'learning_rate': 0.011,
    'subsample': 0.82, 'colsample_bytree': 0.81,
    'min_child_weight': 3, 'gamma': 0.011,
    'reg_alpha': 0.12, 'reg_lambda': 0.4,
    'max_delta_step': 1, 'colsample_bylevel': 0.86,
    'colsample_bynode': 0.88, 'scale_pos_weight': 0.36,
    'max_bin': 512, 'tree_method': 'hist', "device":"cuda",
    'eval_metric': 'rmse', 'random_state': 42,
}

# Run cross-validation
cv_results = xgb.cv(
    params=xgb_params,
    dtrain=dtrain,
    nfold=5,
    num_boost_round=2000,
    metrics='rmse',
    verbose_eval=100,
    early_stopping_rounds=50
)

# Display last few CV results
print(cv_results.tail())

# Extract best boosting round
best_round = cv_results['test-rmse-mean'].idxmin()
best_rmse = cv_results['test-rmse-mean'][best_round]
print(f"Best round: {best_round}, Best CV RMSE: {best_rmse:.7f}")

[0]	train-rmse:0.16524+0.00005	test-rmse:0.16525+0.00019
[100]	train-rmse:0.07642+0.00017	test-rmse:0.07671+0.00021
[200]	train-rmse:0.05822+0.00009	test-rmse:0.05878+0.00024
[300]	train-rmse:0.05563+0.00006	test-rmse:0.05632+0.00024
[400]	train-rmse:0.05528+0.00006	test-rmse:0.05601+0.00023
[500]	train-rmse:0.05521+0.00005	test-rmse:0.05597+0.00023
[600]	train-rmse:0.05518+0.00005	test-rmse:0.05597+0.00023
[700]	train-rmse:0.05517+0.00005	test-rmse:0.05596+0.00023
[800]	train-rmse:0.05517+0.00005	test-rmse:0.05596+0.00023
[900]	train-rmse:0.05517+0.00005	test-rmse:0.05596+0.00023
[1000]	train-rmse:0.05517+0.00005	test-rmse:0.05596+0.00023
[1100]	train-rmse:0.05517+0.00005	test-rmse:0.05596+0.00023
[1200]	train-rmse:0.05517+0.00005	test-rmse:0.05596+0.00023
[1300]	train-rmse:0.05517+0.00005	test-rmse:0.05596+0.00023
[1400]	train-rmse:0.05517+0.00005	test-rmse:0.05596+0.00023
[1475]	train-rmse:0.05517+0.00005	test-rmse:0.05596+0.00023
      train-rmse-mean  train-rmse-std  test-rmse-mea

In [8]:
# putting the n_estimator at the average early stopping point to avoid overfitting
last_round = len(cv_results) - 1
xgb_params["n_estimators"] = last_round + 10

In [9]:
# Prepare training data
X_train = df.drop(columns=target)
y_train = df[target]

# Train XGBoost model
model = XGBRegressor(**xgb_params, enable_categorical=True)
model.fit(X_train, y_train)

# Predict on test set
pred = model.predict(df_test.drop(columns = "id"))

# Prepare submission
sub = pd.DataFrame({
    "id": df_test["id"],
    target: pred
})

# Save submission file
sub.to_csv("xgb_submission.csv", index=False)